# 04_Inhaltsanalyse_Gesicht_Emotion.ipynb

## Inhaltsanalyse (Gesicht & Emotion)

### Ziel
Die Frames sollen auf **menschliche Gesichter** analysiert werden, um sowohl deren **Anzahl** als auch die **emotionale Verfassung** zu quantifizieren.  
Dadurch lassen sich Aussagen über die Präsenz und Stimmung der Personen im Video treffen.

---

## Zu erstellende Features

| Feature | Beschreibung | Wertebereich |
|----------|---------------|---------------|
| **avg_gesichter_pro_frame** | Durchschnittliche Anzahl erkannter Gesichter pro analysiertem Frame | numerisch |
| **dominante_emotion** | Über das gesamte Video am häufigsten erkannte Emotion | 'happy', 'neutral', 'sad', 'angry', 'fear', 'surprise', 'disgust' |

---

## Algorithmus

### Bibliothek: **DeepFace**

`deepface` ist ein Framework, das mehrere modernste Modelle integriert:

1. **Gesichtsdetektion:**  
   - z. B. mit **OpenCV DNN** oder **MTCNN**  
   - dient dem Finden der Gesichter in einem Frame.

2. **Emotionsklassifikation:**  
   - verwendet ein **Convolutional Neural Network (CNN)**,  
   - trainiert auf dem **FER-2013** Datensatz,  
   - erkennt Emotionen wie:  
     `['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']`.

---

## Prozess

1. **Installation der deepface Bibliothek**
2. Laden der bestehenden Features
	- Öffne die Datei features/video_features.csv, die bereits Felder wie
schnitt_frequenz und ist_person_prominent enthält.
	3. Iterieren über jedes Video
	- Für jede video_id:
	- Wähle 3–5 Beispielframes (wie in T3_03_Objekterkennung).
4.	Analyse pro Frame
	-	DeepFace.analyze() für jeden Frame aufrufen:
    -	Anzahl der Gesichter zählen (len(results)).
	-	Emotionen auslesen (results[i]['dominant_emotion']).

5.	Berechnung der neuen Features
	-	avg_gesichter_pro_frame: Durchschnittliche Anzahl erkannter Gesichter pro Frame.
	-	dominante_emotion: Am häufigsten erkannte Emotion über alle Frames.
6.	Speichern der Ergebnisse
	-	Die neuen Features (avg_gesichter_pro_frame, dominante_emotion) zur CSV hinzufügen.
	-	Datei aktualisiert speichern:

## 1. Setup & Import

In [1]:
import os
import pandas as pd
import glob
from deepface import DeepFace
import numpy as np
from collections import defaultdict
import warnings

# --- Konfiguration ---

# 1. Ein- & Ausgabe: Die CSV, die wir kontinuierlich anreichern
FEATURE_FILE = "features/video_features.csv"

# 2. Eingabe: Der Ordner mit den Frames
FRAME_DIR = "data/processed/video_frames"

# 3. Analyse-Parameter
# (Sollte identisch zu T3_03 sein, um die Features konsistent zu halten)
FRAMES_TO_ANALYZE_PER_VIDEO = 5

# Deaktiviert Warnungen von deepface (z.B. Püber Tensorflo-Optimierungen)
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

/Users/zhanna97yan/Files/Viralitaetsanalyse/.venv/lib/python3.10/site-packages/mtcnn/mtcnn.py:34: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


# Methodenwahl: EasyOCR vs. Tesseract

Der ursprüngliche Plan von Gemini 2.5 Pro sah die Nutzung von pytesseract vor. Dies wurde verworfen.

- Problem (Tesseract):
  - Erfordert eine externe Systemabhängigkeit:
    ```bash
    brew install tesseract
    ```
  - Dies verletzt das Prinzip der Portabilität. Das Projekt muss "out-of-the-box" lauffähig sein (nur mit `pip install -r requirements.txt`).

- Lösung (EasyOCR):
  - easyocr ist eine Python-native Bibliothek, die ihre Abhängigkeiten (PyTorch) selbst über pip verwaltet.
  - Sie ist zudem oft robuster bei der Erkennung von stilisierten TikTok-Schriftarten.

## 2. Laden der Daten
Wir laden die CSV-Datei, die von den vorherigen Notebooks erstellt und angereichert wurde.

In [2]:
# Lade die CSV-Datei
try:
    df = pd.read_csv(FEATURE_FILE)
    print(f"{len(df)} Videos aus {FEATURE_FILE} geladen.")
except FileNotFoundError:
    print(f"FEHLER: {FEATURE_FILE} nicht gefunden!")
    print("Stelle sicher, dass T3_02 und T3_03 erfolgreich durchgelaufen sind.")
    raise

# Neue Spalten initialisieren (falls sie nicht schon existieren)
if 'avg_gesichter_pro_frame' not in df.columns:
    df['avg_gesichter_pro_frame'] = 0.0
if 'dominante_emotion' not in df.columns:
    df['dominante_emotion'] = 'none' # 'none' als Standardwert

df.head()

197 Videos aus features/video_features.csv geladen.


,video_id,schnitt_frequenz,durchschnittliche_bewegung,anzahl_frames,video_dauer_sek,ist_person_prominent,ist_tier_sichtbar,avg_objekte_pro_frame,avg_gesichter_pro_frame,dominante_emotion
0,top_53_likes_729700_id_7542648831586880823,0.214953,9.150545,107,107.0,1,0,1.8,0.0,none
1,top_45_likes_780200_id_7548598130665622804,0.192308,6.863052,26,26.0,1,0,1.8,0.0,none
2,top_96_likes_365300_id_7560113050100043026,0.290909,5.528583,55,55.0,1,0,1.6,0.0,none
3,top_32_likes_1000000_id_7556001068405050638,0.110672,9.759456,253,253.0,1,0,1.0,0.0,none
4,top_19_likes_1500000_id_7231352152743152942,0.000000,3.906820,29,29.0,1,0,2.0,0.0,none


## 3. Hauptverarbeitung: Gesichts- & Emotionsanalyse
Wichtiger Hinweis: Wenn diese Zelle das erste Mal läuft, wird deepface die notwendigen vortrainierten Modelle (ca. 50-100 MB) herunterladen. Dies kann einige Minuten dauern. Die Zelle wird lange laufen (104 Minuten hier gedauert)!

In [3]:
print("Starte Gesichts- & Emotionsanalyse für alle Videos (mit MTCNN)...")

# Iteriere durch jede Zeile (jedes Video) im DataFrame
for index, row in df.iterrows():
    video_id = row['video_id']
    
    # 4a. Finde die Frames für dieses Video
    frame_files_pattern = os.path.join(FRAME_DIR, f"{video_id}_frame_*.jpg")
    video_frames = glob.glob(frame_files_pattern)
    
    if not video_frames:
        continue 
        
    # 4b. Wähle Frames für die Analyse aus (Sampling)
    if len(video_frames) > FRAMES_TO_ANALYZE_PER_VIDEO:
        indices = np.linspace(0, len(video_frames) - 1, FRAMES_TO_ANALYZE_PER_VIDEO, dtype=int)
        frames_to_process = [video_frames[i] for i in indices]
    else:
        frames_to_process = video_frames
        
    total_faces_detected = 0
    emotion_counts = defaultdict(int)
    
    # 4c. Führe DeepFace.analyze auf den ausgewählten Frames aus
    for frame_path in frames_to_process:
        try:
            # --- HIER IST DIE ÄNDERUNG ---
            analysis_results = DeepFace.analyze(
                img_path=frame_path, 
                actions=['emotion'], 
                enforce_detection=False,
                detector_backend='mtcnn', # MTCNN ist langsamer, aber viel genauer
                silent=True 
            )
            # --- ENDE DER ÄNDERUNG ---
            
            total_faces_detected += len(analysis_results)
            
            for face_result in analysis_results:
                dominant_emo = face_result['dominant_emotion']
                emotion_counts[dominant_emo] += 1
                
        except Exception as e:
            pass 

    # 4d. Berechne die finalen Features für das Video
    num_analyzed = len(frames_to_process)
    avg_gesichter_pro_frame = total_faces_detected / num_analyzed if num_analyzed > 0 else 0
    
    if not emotion_counts:
        dominante_emotion = 'none'
    else:
        dominante_emotion = max(emotion_counts, key=emotion_counts.get)
    
    # 4e. Speichere Features zurück in den DataFrame
    df.loc[index, 'avg_gesichter_pro_frame'] = avg_gesichter_pro_frame
    df.loc[index, 'dominante_emotion'] = dominante_emotion

    if (index + 1) % 20 == 0:
        print(f"Fortschritt: {index + 1} / {len(df)} Videos verarbeitet.")

print("\n--- Gesichts- & Emotionsanalyse abgeschlossen ---")

Starte Gesichts- & Emotionsanalyse für alle Videos (mit MTCNN)...
1/1 [==============================] - 0s 12ms/step
Fortschritt: 20 / 197 Videos verarbeitet.
1/1 [==============================] - 0s 39ms/step
Fortschritt: 40 / 197 Videos verarbeitet.
15/15 [==============================] - 0s 14ms/step
Fortschritt: 60 / 197 Videos verarbeitet.
1/1 [==============================] - 0s 20ms/step
Fortschritt: 80 / 197 Videos verarbeitet.
1/1 [==============================] - 0s 14ms/step
Fortschritt: 100 / 197 Videos verarbeitet.
1/1 [==============================] - 0s 12ms/step
Fortschritt: 120 / 197 Videos verarbeitet.
1/1 [==============================] - 0s 13ms/step
Fortschritt: 140 / 197 Videos verarbeitet.
1/1 [==============================] - 0s 47ms/step
Fortschritt: 160 / 197 Videos verarbeitet.
11/11 [==============================] - 0s 13ms/step
Fortschritt: 180 / 197 Videos verarbeitet.
1/1 [==============================] - 0s 24ms/step

--- Gesichts- & Emotionsan

## 4. Ergebnis speichern
Die video_features.csv wird nun mit den neuen Spalten überschrieben.

In [4]:
# 5. Ergebnisse in dieselbe CSV-Datei zurückspeichern
try:
    df.to_csv(FEATURE_FILE, index=False)
    print(f"Erfolgreich aktualisiert: {FEATURE_FILE}")
    
    # Zeige die neuen Spalten in der Vorschau
    print("\nAktualisierte Datei-Vorschau (video_features.csv):")
    cols_to_show = [
        'video_id', 
        'ist_person_prominent', 
        'avg_gesichter_pro_frame', 
        'dominante_emotion'
    ]
    # Filtere Spalten, die auch wirklich existieren (falls ein Schritt übersprungen wurde)
    existing_cols_to_show = [col for col in cols_to_show if col in df.columns]
    print(df[existing_cols_to_show].head())
    
except PermissionError:
    print(f"\nFEHLER: Keine Berechtigung, {FEATURE_FILE} zu schreiben.")
    print("Ist die Datei vielleicht in Excel oder einem anderen Programm geöffnet?")
except Exception as e:
    print(f"\nEin Fehler ist beim Speichern aufgetreten: {e}")

Erfolgreich aktualisiert: features/video_features.csv

Aktualisierte Datei-Vorschau (video_features.csv):
                                      video_id  ist_person_prominent  \
0   top_53_likes_729700_id_7542648831586880823                     1   
1   top_45_likes_780200_id_7548598130665622804                     1   
2   top_96_likes_365300_id_7560113050100043026                     1   
3  top_32_likes_1000000_id_7556001068405050638                     1   
4  top_19_likes_1500000_id_7231352152743152942                     1   

   avg_gesichter_pro_frame dominante_emotion  
0                      1.6             happy  
1                      1.0             happy  
2                      1.2               sad  
3                      1.0             happy  
4                      1.6             happy  


## Erkenntnis

- Die Kombination von **Text-Sentiment** (`text_sentiment_compound`) und **dominante_emotion** (Gesicht) ist **vielversprechend**.  
- Beispiel: Videos mit **traurigen Gesichtern** (`dominante_emotion = 'sad'`) aber **positivem Text** (`text_sentiment_compound > 0.5`) könnten eine besondere **Viralität** aufweisen.